In [1]:
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
#from tensorflow.keras.applications.inception_v3 import InceptionV3              #M-1
#from tensorflow.keras.applications.vgg16 import VGG16                             #M-2
#from tensorflow.keras.applications.vgg19 import VGG19                              #M-5
#from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2                #M-3
from tensorflow.keras.applications.resnet50 import ResNet50                       #m-4
#from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.applications.resnet50 import preprocess_input  
#from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
#from tensorflow.keras.applications.vgg19 import preprocess_input
#from tensorflow.keras.applications.vgg16 import preprocess_input           
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adadelta, Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
#resize all the images to this:

IMAGE_SIZE =[75, 100]
train_path = ''
valid_path = ''

In [3]:
resnet = ResNet50(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False, pooling = 'avg')

94773248/94765736 [==============================] - 1s 0us/step


In [4]:
resnet.summary()

Model: "resnet50"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 75, 100, 3)] 0                                            
__________________________________________________________________________________________________
conv1_pad (ZeroPadding2D)       (None, 81, 106, 3)   0           input_1[0][0]                    
__________________________________________________________________________________________________
conv1_conv (Conv2D)             (None, 38, 50, 64)   9472        conv1_pad[0][0]                  
__________________________________________________________________________________________________
conv1_bn (BatchNormalization)   (None, 38, 50, 64)   256         conv1_conv[0][0]                 
___________________________________________________________________________________________

In [ ]:
folders = glob('/*')
folders

In [ ]:
for layer in resnet.layers:
    layer.trainable = False

for layer in resnet.layers[-22:]:
    layer.trainable = True

In [ ]:
from tensorflow.keras.regularizers import l2
import tensorflow as tf
x = Dropout(rate = 0.5)(resnet.output)
x = Dense(128, activation="relu",kernel_regularizers=tf.keras.regularizers.l2(0.02))(x)
x = Dropout(0.5)(x)

In [ ]:
prediction = Dense(len(folders),  kernel_regularizer=tf.keras.regularizers.l2(0.02), activation='softmax')(x)

In [ ]:
#create a model object
model = Model(inputs = inceptionv3.input, outputs = prediction)

In [ ]:
#model summary
model.summary()

In [ ]:
# Define the optimizer
optimizer = Adam(lr=0.001)

In [ ]:
# Compile the model
model.compile(optimizer = optimizer , 
              loss = "categorical_crossentropy", 
              metrics=["accuracy"])

In [ ]:
# Set a learning rate annealer
learning_rate_reduction = ReduceLROnPlateau(monitor='val_accuracy', 
                                            patience=3, 
                                            verbose=1, 
                                            factor=0.5, 
                                            min_lr=0.00001)

In [ ]:
#use Image data generator to import the images from the dataset
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen =ImageDataGenerator(rescale = 1./255,
                                  featurewise_center=False,  # set input mean to 0 over the dataset
                                  samplewise_center=False,  # set each sample mean to 0
                                  featurewise_std_normalization=False,  # divide inputs by std of the dataset
                                  samplewise_std_normalization=False,  # divide each input by its std
                                  zca_whitening=False,  # apply ZCA whitening
                                  rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
                                  zoom_range = 0.1, # Randomly zoom image 
                                  width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
                                  height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
                                  horizontal_flip=False,  # randomly flip images
                                  vertical_flip=False
                                )
test_datagen =ImageDataGenerator(rescale = 1./255)

In [ ]:
training_set = train_datagen.flow_from_directory('',
                                                 target_size = (75, 100),
                                                 batch_size = 10,
                                                 class_mode = 'categorical')

In [ ]:
test_set = test_datagen.flow_from_directory('',
                                                 target_size = (75, 100),
                                                 batch_size = 10,
                                                 class_mode = 'categorical')

In [ ]:
#fit the model
r = model.fit_generator(
    training_set,
    validation_data = test_set,
    epochs = 30,
    steps_per_epoch = len(training_set),
    validation_steps = len(test_set),
    callbacks = [learning_rate_reduction]
)

In [ ]:
plt.plot(r.history['loss'], label = 'train loss')
plt.plot(r.history['val_loss'], label = 'val loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('no. of epochs')
plt.legend()
plt.show()

In [ ]:
plt.plot(r.history['accuracy'], label = 'train acc')
plt.plot(r.history['val_accuracy'], label = 'val acc')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('no. of epochs')
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.models import load_model
model.save ('model_resnet.h5')

In [ ]:
y_pred = model.predict(test_set)

In [ ]:
print(y_pred)

In [ ]:
import numpy as np
y_pred = np.argmax(y_pred, axis=1)
print(y_pred)

In [ ]:
#confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix
#print('Confusion Matrix')
#dict_characters = {0: "akiec", 1: "bcc", 2: "bkl", 3: "df", 4: "mel", 5: "nv", 6: "vasc"}
confusion_matrix= confusion_matrix(test_set.classes, y_pred)
plt.title('Confusion Matrix')
plt.ylabel("Actual")
plt.xlabel("Predicted")
sns.heatmap(confusion_matrix, annot = True, fmt = 'g' ,vmin = 0, cmap = 'Blues')
#print(confusion_matrix)

In [ ]:
from sklearn.metrics import classification_report
print('Classification Report')
target_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
print(classification_report(test_set.classes, y_pred, target_names=target_names))